# a2ui-apps composer — QLoRA fine-tune (Kaggle, free T4)

Distills the freeform pathway's compose step (A2UI Express against the pruned cre8 catalog) into a small open model, then pushes it to Hugging Face for Modal (or Ollama) to serve. Data comes from `scripts/distill.py`; the real check is `scripts/eval_composer.py` afterwards.

**Before you run (one time):**
1. *Settings → Accelerator:* **GPU T4 x2** (one is used). *Settings → Internet:* **On** (needs a phone-verified Kaggle account).
2. *Add-ons → Secrets:* add `HF_TOKEN` (a Hugging Face **write** token) and attach it to this notebook.
3. *Add Input:* your dataset containing `train.jsonl` and `val.jsonl` (`training/kaggle/push.sh` uploads it for you).
4. Edit `HF_REPO` below. Then **Save Version → Save & Run All** so it runs headless (up to 12 h) without the tab open.

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
# ---- settings ---------------------------------------------------------------
BASE_MODEL  = "unsloth/Qwen3-4B-Instruct-2507"   # text-only, no <think>; fits a T4 with room for long JSON
HF_REPO     = "YOUR_HF_USERNAME/a2ui-apps-composer-4b"   # merged 16-bit model goes here (private)
MAX_SEQ     = 8192     # slim prompt (~2-5k tokens) + Express reply; longer examples are dropped
EPOCHS      = 2
LR          = 2e-4
LORA_RANK   = 32
BATCH       = 1
GRAD_ACCUM  = 16       # effective batch 16; long sequences need batch 1 on a 16 GB T4
PUSH_GGUF   = True     # also push a Q4_K_M GGUF so you can run it locally in Ollama for $0

In [ ]:
import glob
import json
import os
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert "YOUR_HF_USERNAME" not in HF_REPO, "Set HF_REPO first"

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input — did you attach the dataset?"
    return hits[0]

DATA_DIR = os.path.dirname(find("train.jsonl"))

def load(p): return [json.loads(line) for line in open(p)]
train_rows, val_rows = load(f"{DATA_DIR}/train.jsonl"), load(f"{DATA_DIR}/val.jsonl")
print(len(train_rows), "train /", len(val_rows), "val")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,            # auto: fp16 on T4
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from datasets import Dataset

def to_text(rows):
    out, dropped, lens = [], 0, []
    for r in rows:
        text = tokenizer.apply_chat_template(r["messages"], tokenize=False)
        n = len(tokenizer(text, add_special_tokens=False)["input_ids"])
        if n > MAX_SEQ:
            dropped += 1
            continue
        lens.append(n)
        out.append({"text": text})
    lens.sort()
    print(f"kept {len(out)}, dropped {dropped} over {MAX_SEQ} tokens; "
          f"median {lens[len(lens)//2] if lens else 0}, p95 {lens[int(len(lens)*.95)] if lens else 0}")
    return Dataset.from_list(out)

train_ds, val_ds = to_text(train_rows), to_text(val_rows)
print(train_ds[0]["text"][:600])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds if len(val_ds) else None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps" if len(val_ds) else "no",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        output_dir="/kaggle/working/ckpt",
        seed=3407,
        report_to="none",
    ),
)
# Loss only on the assistant JSON, not the system prompt or request.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

## Smoke test — the real validity check is `scripts/eval_composer.py` against the served model

In [ ]:
FastLanguageModel.for_inference(model)

def compose(messages, max_new_tokens=3000):
    ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.2, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

looks_right = 0
for r in val_rows[:5]:   # ~1-2 min each on a T4
    reply = compose(r["messages"][:2])
    ok = "<a2ui>" in reply and "</a2ui>" in reply and "root =" in reply
    looks_right += ok
    print("OK " if ok else "?? ", r["messages"][1]["content"][:80])
    print(reply[:700])
    print("-" * 60)
print(f"{looks_right}/5 well-formed Express blocks")

## Push to Hugging Face (private)

In [ ]:
model.save_pretrained("/kaggle/working/lora")          # adapter, kept as notebook output too
tokenizer.save_pretrained("/kaggle/working/lora")
model.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN, private=True)

# Merged 16-bit weights: what vLLM on Modal loads.
model.push_to_hub_merged(HF_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN, private=True)
print("pushed", HF_REPO)

In [ ]:
# Optional: GGUF for Ollama on your own machine (slow step: builds llama.cpp).
if PUSH_GGUF:
    model.push_to_hub_gguf(HF_REPO + "-GGUF", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN, private=True)
    print("ollama run hf.co/" + HF_REPO + "-GGUF:Q4_K_M")